Name: Akhlak Hossain

Student ID: 2022-3-60-057

Date: August 5, 2026

In [16]:
!apt-get update
!apt-get install -y wget
!wget -q -O - https://dl-ssl.google.com/linux/linux_signing_key.pub | apt-key add -
!echo "deb [arch=amd64] http://dl.google.com/linux/chrome/deb/ stable main" >> /etc/apt/sources.list.d/google-chrome.list
!apt-get update
!apt-get install google-chrome-stable -y

!pip install selenium pandas webdriver-manager

Hit:1 http://dl.google.com/linux/chrome/deb stable InRelease
Hit:2 https://cli.github.com/packages stable InRelease
Hit:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:4 https://dl.google.com/linux/chrome-stable/deb stable InRelease
Hit:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:6 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:8 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Hit:11 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Reading package lists... Done
W: http://dl.google.com/linux/chrome/deb/dists/stable/InRelease: Key is stored in legacy trusted.gpg keyring (/etc/apt/trusted.gpg), see the DEPRECATION section in apt-key(8) for details.
W: Skipping acquire of configured file 'main/source/Sources'

In [23]:
import time
import csv
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.service import Service as ChromeService
from webdriver_manager.chrome import ChromeDriverManager

total_assertions = 0
passed_assertions = 0

def log(test_name, passed):
    """Helper function to log test results without crashing the notebook."""
    global total_assertions, passed_assertions
    total_assertions += 1
    if passed:
        passed_assertions += 1
        print(f"[PASS] {test_name}")
    else:
        print(f"[FAIL] {test_name}")

In [24]:
# Task 1: Setup and Navigation

# Configure Chrome options for headless execution in Colab
chrome_options = webdriver.ChromeOptions()
chrome_options.add_argument('--headless')
chrome_options.add_argument('--no-sandbox')
chrome_options.add_argument('--disable-dev-shm-usage')
chrome_options.add_argument('--disable-gpu') # Added for better headless compatibility
# Explicitly set the path to the Chrome binary for google-chrome-stable
chrome_options.binary_location = '/usr/bin/google-chrome'

# Initialize WebDriver using ChromeDriverManager
driver = webdriver.Chrome(service=ChromeService(ChromeDriverManager().install()), options=chrome_options)

# a. Navigate to https://books.toscrape.com
driver.get("https://books.toscrape.com")

# b. Print the page title and current URL
page_title = driver.title
current_url = driver.current_url
print(f"Page Title: {page_title}")
print(f"Current URL: {current_url}")

# c. Maximize the browser window
driver.maximize_window()

# d. Assert that the string "Books" appears in the page title
log('Title contains "Books"', "Books" in page_title)

# e. Assert that the current URL starts with "https://"
log('URL starts with "https://"', current_url.startswith("https://"))

# f. Refresh the page, wait 2 seconds, then assert the title is unchanged
driver.refresh()
time.sleep(2)
log('Title unchanged after refresh', driver.title == page_title)

Page Title: All products | Books to Scrape - Sandbox
Current URL: https://books.toscrape.com/
[PASS] Title contains "Books"
[PASS] URL starts with "https://"
[PASS] Title unchanged after refresh


In [25]:
# Task 2: Locate elements using four different strategies

# a. By.CLASS_NAME: Find all book title elements (usually h3 > a)
titles_by_class = driver.find_elements(By.CLASS_NAME, "product_pod")
print(f"\nFound {len(titles_by_class)} book containers using CLASS_NAME.")
if len(titles_by_class) > 0:
    sample_title = titles_by_class[0].find_element(By.TAG_NAME, "h3").text
    print(f"Sample book title: {sample_title}")
log("Count of elements by CLASS_NAME > 0", len(titles_by_class) > 0)

# b. By.CSS_SELECTOR: Find all star-rating elements
star_ratings = driver.find_elements(By.CSS_SELECTOR, ".star-rating")
print(f"\nFound {len(star_ratings)} star-rating elements using CSS_SELECTOR.")
if len(star_ratings) > 0:
    print(f"Sample star rating class: {star_ratings[0].get_attribute('class')}")
log("Count of elements by CSS_SELECTOR > 0", len(star_ratings) > 0)

# c. By.XPATH: Find the "next" pagination button
next_button = driver.find_elements(By.XPATH, "//li[@class='next']/a")
print(f"\nFound {len(next_button)} 'next' buttons using XPATH.")
if len(next_button) > 0:
    print(f"Sample next button text: {next_button[0].text}")
log("Count of elements by XPATH > 0", len(next_button) > 0)

# d. By.TAG_NAME: Find all <article> elements
articles = driver.find_elements(By.TAG_NAME, "article")
print(f"\nFound {len(articles)} <article> elements using TAG_NAME.")
if len(articles) > 0:
    print(f"Sample article class attribute: {articles[0].get_attribute('class')}")
log("Count of elements by TAG_NAME > 0", len(articles) > 0)


Found 20 book containers using CLASS_NAME.
Sample book title: A Light in the ...
[PASS] Count of elements by CLASS_NAME > 0

Found 20 star-rating elements using CSS_SELECTOR.
Sample star rating class: star-rating Three
[PASS] Count of elements by CSS_SELECTOR > 0

Found 1 'next' buttons using XPATH.
Sample next button text: next
[PASS] Count of elements by XPATH > 0

Found 20 <article> elements using TAG_NAME.
Sample article class attribute: product_pod
[PASS] Count of elements by TAG_NAME > 0


In [20]:
# Task 3: Scrape data from pages 1 through 4
scraped_data = []

for page in range(1, 5):
    url = f"https://books.toscrape.com/catalogue/page-{page}.html"
    driver.get(url)
    time.sleep(1)

    books = driver.find_elements(By.CLASS_NAME, "product_pod")

    for book in books:
        title = book.find_element(By.TAG_NAME, "h3").find_element(By.TAG_NAME, "a").get_attribute("title")
        price = book.find_element(By.CLASS_NAME, "price_color").text
        rating_class = book.find_element(By.CSS_SELECTOR, "p.star-rating").get_attribute("class")
        rating = rating_class.split(" ")[1] if len(rating_class.split(" ")) > 1 else "Unknown"
        availability = book.find_element(By.CLASS_NAME, "instock").text.strip()

        scraped_data.append({
            "page": page,
            "title": title,
            "price": price,
            "rating": rating,
            "availability": availability
        })

# a. Print total number of books and assert it equals 80
total_books = len(scraped_data)
print(f"\nTotal books collected: {total_books}")
log("Total books collected equals 80", total_books == 80)

# b. Save data to CSV
csv_filename = "books_data.csv"
with open(csv_filename, mode='w', newline='', encoding='utf-8') as file:
    writer = csv.DictWriter(file, fieldnames=["page", "title", "price", "rating", "availability"])
    writer.writeheader()
    writer.writerows(scraped_data)

# c. Print first 3 rows using pandas
df = pd.read_csv(csv_filename)
print("\nFirst 3 rows of collected data:")
print(df.head(3))


Total books collected: 80
[PASS] Total books collected equals 80

First 3 rows of collected data:
   page                 title   price rating availability
0     1  A Light in the Attic  £51.77  Three     In stock
1     1    Tipping the Velvet  £53.74    One     In stock
2     1            Soumission  £50.10    One     In stock


In [21]:
# Task 4: Validate content on detail pages for 3 specific books
test_books = scraped_data[:3]

for book in test_books:
    driver.get("https://books.toscrape.com")

    book_link = WebDriverWait(driver, 10).until(
        EC.presence_of_element_located((By.XPATH, f"//a[@title=\"{book['title']}\"]"))
    )
    book_link.click()

    # a. Assert the page title is not empty
    log(f"Detail page title not empty for '{book['title']}'", bool(driver.title.strip()))

    # b. Assert the book title matches
    detail_title = driver.find_element(By.TAG_NAME, "h1").text
    log(f"Detail title matches for '{book['title']}'", book['title'] in detail_title or detail_title in book['title'])

    # c. Assert the price matches
    detail_price = driver.find_element(By.CLASS_NAME, "price_color").text
    log(f"Detail price matches for '{book['title']}'", detail_price == book['price'])

    # d. Assert Add to basket is present and enabled
    add_btn = driver.find_elements(By.CLASS_NAME, "btn-add-to-basket")
    btn_present = len(add_btn) > 0
    btn_enabled = btn_present and add_btn[0].is_enabled()
    log(f"Add to basket button present and enabled for '{book['title']}'", btn_enabled)

    # e. Assert availability contains \"In stock\"
    detail_availability = driver.find_element(By.CLASS_NAME, "instock").text
    log(f"Availability contains 'In stock' for '{book['title']}'", "In stock" in detail_availability)

[PASS] Detail page title not empty for 'A Light in the Attic'
[PASS] Detail title matches for 'A Light in the Attic'
[PASS] Detail price matches for 'A Light in the Attic'
[FAIL] Add to basket button present and enabled for 'A Light in the Attic'
[PASS] Availability contains 'In stock' for 'A Light in the Attic'
[PASS] Detail page title not empty for 'Tipping the Velvet'
[PASS] Detail title matches for 'Tipping the Velvet'
[PASS] Detail price matches for 'Tipping the Velvet'
[FAIL] Add to basket button present and enabled for 'Tipping the Velvet'
[PASS] Availability contains 'In stock' for 'Tipping the Velvet'
[PASS] Detail page title not empty for 'Soumission'
[PASS] Detail title matches for 'Soumission'
[PASS] Detail price matches for 'Soumission'
[FAIL] Add to basket button present and enabled for 'Soumission'
[PASS] Availability contains 'In stock' for 'Soumission'


In [22]:
# Task 5: Final Summary
print("\n" + "="*30)
print(f"FINAL TEST SUMMARY: {passed_assertions} out of {total_assertions} assertions passed.")
print("="*30 + "\n")

# Close the browser instance
driver.quit()


FINAL TEST SUMMARY: 20 out of 23 assertions passed.

